In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from sklearn.metrics import classification_report, f1_score

2025-09-11 11:06:11.950301: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data_dir = "../data/cloud_dataset"

In [3]:
img_size = (128, 128)
batch_size = 32
seed = 123

In [4]:
# Load dataset with 80% train + 20% temp (val+test)
full_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

Found 5304 files belonging to 2 classes.
Using 4244 files for training.


I0000 00:00:1757588784.269211   27839 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1916 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [5]:
temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)


Found 5304 files belonging to 2 classes.
Using 1060 files for validation.


In [6]:
# Split temp_ds (20%) into validation (10%) and test (10%)
val_size = 0.5
val_ds = temp_ds.take(int(len(temp_ds) * val_size))
test_ds = temp_ds.skip(int(len(temp_ds) * val_size))

In [7]:
normalization_layer = layers.Rescaling(1./255)
full_train_ds = full_train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [9]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=img_size + (3,)
)
base_model.trainable = False

In [10]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

In [11]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [12]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [13]:
history = model.fit(
    full_train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stop]
)

Epoch 1/100


2025-09-11 11:08:47.448536: I external/local_xla/xla/service/service.cc:163] XLA service 0x7496400028e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-11 11:08:47.448592: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2025-09-11 11:08:48.187332: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-11 11:08:52.529155: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
I0000 00:00:1757588947.385027   27938 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


133/133 ━━━━━━━━━━━━━━━━━━━━ 79s 316ms/step - accuracy: 0.8977 - loss: 0.2946 - val_accuracy: 0.9301 - val_loss: 0.2004
Epoch 2/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.9343 - loss: 0.1917 - val_accuracy: 0.9283 - val_loss: 0.1762
Epoch 3/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - accuracy: 0.9390 - loss: 0.1676 - val_accuracy: 0.9338 - val_loss: 0.1547
Epoch 4/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.9465 - loss: 0.1533 - val_accuracy: 0.9246 - val_loss: 0.1621
Epoch 5/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9458 - loss: 0.1421 - val_accuracy: 0.9357 - val_loss: 0.1324
Epoch 6/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.9482 - loss: 0.1413 - val_accuracy: 0.9375 - val_loss: 0.1339
Epoch 7/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9486 - loss: 0.1324 - val_accuracy: 0.9338 - val_loss: 0.1377
Epoch 8/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 9s 65ms/step - accuracy: 0.9491 - loss: 0.1315 - val_accu

In [78]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.4f}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9399 - loss: 0.1536
Test Accuracy: 0.9399


In [79]:
# Collect all test images and labels in one pass
x_test, y_true = [], []
for x, y in test_ds:
    x_test.append(x.numpy())
    y_true.append(y.numpy())

x_test = np.concatenate(x_test, axis=0)
y_true = np.concatenate(y_true, axis=0)

# Predict
y_pred_probs = model.predict(x_test)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

# F1 score
f1 = f1_score(y_true, y_pred)
print(f"Test F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))


17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 253ms/step
Test F1-score: 0.9445

Classification Report:
              precision    recall  f1-score   support

           0     0.8821    0.9712    0.9245       208
           1     0.9791    0.9123    0.9445       308

    accuracy                         0.9360       516
   macro avg     0.9306    0.9417    0.9345       516
weighted avg     0.9400    0.9360    0.9365       516



In [14]:
model.save("../models/DenseNet_model.keras")

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# Load model
model = tf.keras.models.load_model("../models/DenseNet_model.keras")

# Load and preprocess image
img_path = "test/CB1.jpg"
img_size = (128, 128)

img = image.load_img(img_path, target_size=img_size)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict probability
prob = model.predict(img_array)[0][0]

# Convert probability → class (0 or 1)
pred_class = int(prob > 0.5)

print(f"Predicted probability: {prob:.4f}")
print(f"Predicted class: {pred_class}")


2025-09-12 05:43:34.252847: I external/local_xla/xla/service/service.cc:163] XLA service 0x7283f4003320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-12 05:43:34.252886: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2025-09-12 05:43:34.477719: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-12 05:43:36.472274: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


1/1 ━━━━━━━━━━━━━━━━━━━━ 19s 19s/step
Predicted probability: 1.0000
Predicted class: 1


I0000 00:00:1757655829.644151   12605 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
